# ĐỒ ÁN MÔN HỌC — NT120: NHẬP MÔN TRÍ TUỆ NHÂN TẠO CHO MẠNG MÁY TÍNH VÀ AN TOÀN THÔNG TIN

## Phân loại Thiết bị IoT trong Mạng Smart Home bằng Machine Learning dựa trên Đặc tính Lưu lượng Mạng

**Thông tin nhóm**

| Mục | Nội dung |
|---|---|
| Lớp | NT120.R12 |
| Nhóm số | 05 |
| Giảng viên hướng dẫn | ThS. Huỳnh Văn Đặng, TS. Nguyễn Khánh Thuật |
| Học kỳ | Học kỳ 1 — Năm học 2026-2027 |
| Ngày nộp | 15/10/2026 |

**Danh sách thành viên**

| STT | MSSV | Họ và tên | Vai trò / Phần đóng góp chính | Email |
|---|---|---|---|---|
| 1 | 22520001 | Nguyễn Văn A | Trưởng nhóm: Thu thập dữ liệu, Tiền xử lý & Feature Engineering | 22520001@gm.uit.edu.vn |
| 2 | 22520002 | Trần Thị B | Huấn luyện mô hình, Đánh giá Benchmark & Rút gọn đặc trưng | 22520002@gm.uit.edu.vn |
| 3 | 22520003 | Lê Hoàng C | Xây dựng ứng dụng Web Dashboard & Viết báo cáo tổng kết | 22520003@gm.uit.edu.vn |

## Tóm tắt Đồ án (Executive Summary)
Đồ án giải quyết bài toán **Tự động nhận diện và phân loại thiết bị IoT (IoT Device Classification)** trong mạng Smart Home bằng các giải thuật học máy (Machine Learning). Hệ thống hoạt động hoàn toàn dựa trên **đặc tính thống kê lưu lượng gói tin (Packet Metadata & Traffic Characteristics)** thu thập tại cổng Router Gateway mà **hoàn toàn không cần đọc nội dung gói tin (Payload)**, đảm bảo tính khả thi ngay cả khi lưu lượng được mã hóa bảo mật (TLS/HTTPS).
- **Bộ dữ liệu thực nghiệm:** UNSW Smart Home IoT Traffic Dataset (IEEE TMC 2018) với hơn 2.3 triệu gói tin của 20 dòng thiết bị thực tế.
- **Phương pháp luận:** Gom cửa sổ thời gian 10 giây trích xuất 15 đặc trưng mạng; áp dụng phân chia theo trục thời gian (**Time-based Split**) triệt tiêu rò rỉ dữ liệu (Zero Data Leakage).
- **Kết quả:** Các mô hình Ensemble (Random Forest, XGBoost) đạt độ chính xác ấn tượng **~97.8%**, Macro F1 **~79.7%**. Mô hình **Decision Tree** chứng minh tính tối ưu cho thiết bị Router biên (Ultra-Edge) với độ trễ suy diễn cực thấp chỉ **0.36 µs/mẫu** và dung lượng siêu nhẹ **294 KB**.

## Mục lục

1. Giới thiệu và phát biểu bài toán
2. Khảo sát các giải pháp liên quan
3. Thiết lập môi trường và nạp dữ liệu
4. Tiền xử lý dữ liệu
5. Phương pháp tiếp cận
6. Thực nghiệm và đánh giá kết quả
7. Phân tích và thảo luận
8. Kết luận và hướng phát triển
9. Đóng góp của các thành viên
10. Tài liệu tham khảo
11. Phụ lục (tùy chọn)
12. Checklist tự đánh giá trước khi nộp
13. Gợi ý chuẩn bị phần trình bày trước lớp

## 1. Giới thiệu và phát biểu bài toán

### 1.1 Bối cảnh và động lực
Sự bùng nổ của cuộc cách mạng Internet of Things (IoT) đã đưa hàng tỷ thiết bị thông minh (camera giám sát, loa thông minh, cảm biến môi trường, thiết bị y tế gia đình...) kết nối vào mạng Smart Home và mạng doanh nghiệp. Tuy nhiên, tình trạng này gây ra thách thức to lớn cho công tác quản trị và an toàn thông tin:
1. **Kiểm kê thiết bị (Device Inventory) bất khả thi thủ công:** Quản trị viên mạng không thể theo dõi liên tục hàng chục thiết bị ra vào hệ thống mạng nội bộ.
2. **Nguy cơ an ninh từ thiết bị bị thỏa hiệp (Compromised IoT):** Thiết bị IoT thường có năng lực tính toán hạn chế, firmware ít được vá lỗi, dễ dàng bị tin tặc biến thành botnet (như cuộc tấn công Mirai kỷ lục).
3. **Thách thức từ mã hóa lưu lượng (Encrypted Traffic):** Ngày nay, phần lớn thiết bị IoT sử dụng giao thức HTTPS/TLS hoặc cơ chế bảo mật đám mây riêng, khiến phương pháp kiểm tra sâu gói tin (Deep Packet Inspection - DPI) truyền thống trở nên vô hiệu và vi phạm quyền riêng tư.
Vì vậy, việc ứng dụng Trí tuệ Nhân tạo để phân loại thiết bị IoT chỉ dựa trên **dấu vân tay lưu lượng mạng (Traffic Fingerprinting)** là hướng tiếp cận cấp thiết và có giá trị ứng dụng cao.

### 1.2 Phát biểu bài toán
- **Đầu vào (Input):** Chuỗi gói tin mạng thu thập tại cổng Gateway của mạng LAN (bao gồm timestamp, kích thước gói, địa chỉ MAC nguồn/đích, giao thức IP.proto, số hiệu cổng port.src/port.dst).
- **Đầu ra (Output):** Nhãn nhận diện thiết bị IoT, được phân loại theo:
  1. *Phân loại chi tiết (Fine-grained):* 20 dòng thiết bị cụ thể (Amazon Echo, Dropcam, Smart Things, Withings Baby Monitor, Belkin Wemo Switch...).
  2. *Phân loại theo danh mục (Coarse-grained):* 8 nhóm chức năng (Camera, Smart Plug/Switch, Sensor/Alarm, Health/Wellness, Audio/Media, Smart Light, Hub, Non-IoT).
- **Dạng bài toán:** Học máy có giám sát (Supervised Learning) — Phân loại đa lớp (Multi-class Classification).
- **Giả định (Assumptions):**
  - Lưu lượng mạng được theo dõi tập trung tại Router Gateway trung tâm (nơi mọi gói tin vào/ra Internet đều đi qua).
  - Không giải mã payload (chấp nhận gói tin bị mã hóa đầu cuối).

### 1.3 Mục tiêu cụ thể
1. Xây dựng quy trình tiền xử lý tự động: Gom cụm gói tin theo cửa sổ thời gian 10 giây và trích xuất 15 đặc trưng thống kê lưu lượng có khả năng phân biệt cao.
2. Áp dụng chiến lược chia tập theo thời gian thực tế (**Time-based Split**) để kiểm thử khả năng tổng quát hóa trên dữ liệu ngày mới.
3. Huấn luyện, đánh giá và so sánh đa chiều **7 mô hình** (bao gồm 2 đường cơ sở Baseline và 5 giải thuật ML: Decision Tree, Random Forest, XGBoost, KNN, MLP).
4. Đạt độ chính xác tổng thể (Accuracy) trên tập kiểm tra $\ge 95\%$ và Macro F1-score $\ge 75\%$.
5. Phân tích sự đánh đổi kỹ thuật (Engineering Trade-off) giữa độ chính xác và độ trễ suy diễn để tìm giải pháp tối ưu cho triển khai trên thiết bị Router biên.
6. Xây dựng ứng dụng Web Dashboard thực tế minh họa khả năng kiểm kê thiết bị và phát hiện thiết bị lạ (Rogue Device Detection).

### 1.4 Phạm vi và giới hạn
- **Phạm vi dữ liệu:** Thực nghiệm trên bộ dữ liệu UNSW Smart Home IoT Traffic chuẩn quốc tế (thu thập trong môi trường Smart Home thực tế tại Úc).
- **Giới hạn kỹ thuật:** Đồ án không giải mã nội dung gói tin; các thiết bị Bluetooth/Zigbee không kết nối trực tiếp qua Wi-Fi/Ethernet sẽ được nhận diện gián tiếp thông qua IoT Gateway Hub (như SmartThings Hub).
- **Môi trường thử nghiệm:** Huấn luyện trên máy tính trạm và đánh giá mô phỏng triển khai nhúng trên Router dựa trên đo đạc Microsecond Latency.

## 2. Khảo sát các giải pháp liên quan

Nghiên cứu kế thừa và đối sánh với 3 công trình tiêu biểu trong lĩnh vực IoT Traffic Classification:

1. **Sivanathan et al. (IEEE TMC 2018) [1]:**
   - *Vấn đề giải quyết:* Phân loại thiết bị IoT trong môi trường Smart Home sử dụng đặc tính lưu lượng mạng.
   - *Cách tiếp cận:* Thu thập lưu lượng mạng của 28 thiết bị IoT trong 6 tháng tại UNSW Sydney. Trích xuất đặc trưng thống kê luồng (flow volume, duration, port) và áp dụng Random Forest và Naive Bayes.
   - *Điểm kế thừa & phát triển:* Nhóm kế thừa bộ dữ liệu và định hướng phân loại không đọc payload của bài báo; phát triển thêm kỹ thuật gom cửa sổ thời gian 10s cố định và so sánh sâu về độ trễ Edge Router vs Cloud Server.

2. **Miettinen et al. (IEEE S&P 2017) — IoT Sentinel [2]:**
   - *Vấn đề giải quyết:* Tự động nhận diện thiết bị IoT khi vừa kết nối vào mạng (Device Fingerprinting during onboarding).
   - *Cách tiếp cận:* Trích xuất 23 đặc trưng từ chuỗi 12 gói tin đầu tiên khi thiết bị kết nối mạng (Setup phase) và phân loại bằng Random Forest.
   - *So sánh:* IoT Sentinel chỉ nhận diện được lúc thiết bị khởi động; giải pháp của nhóm mở rộng nhận diện liên tục trong suốt quá trình hoạt động (Continuous Operation).

3. **Meidan et al. (IEEE IoT Journal 2017) — ProfilIoT [3]:**
   - *Vấn đề giải quyết:* Phân biệt thiết bị IoT với thiết bị thông thường (Non-IoT).
   - *Cách tiếp cận:* Áp dụng học máy đa lớp để lập hồ sơ lưu lượng mạng của từng thiết bị.
   - *Kế thừa:* Nhóm tích hợp bài toán phân loại phân cấp: vừa nhận diện IoT vs Non-IoT, vừa phân loại 20 model thiết bị cụ thể.

## 3. Thiết lập môi trường và nạp dữ liệu

### 3.1 Cài đặt và nạp thư viện
Import các thư viện xử lý dữ liệu, học máy và trực quan hóa. Cố định `SEED = 42` để đảm bảo kết quả tái lập tuyệt đối.

In [ ]:
# Thư viện tính toán và xử lý dữ liệu
import os
import json
import time
import numpy as np
import pandas as pd

# Thư viện trực quan hóa
import matplotlib.pyplot as plt
import seaborn as sns

# Thư viện Machine Learning (scikit-learn & xgboost)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Cố định random seed để kết quả tái lập 100%
SEED = 42
np.random.seed(SEED)
print("[✓] Thư viện đã nạp thành công! Random Seed cố định:", SEED)

### 3.2 Nguồn dữ liệu
Dữ liệu sử dụng là bộ dữ liệu thực tế **UNSW Smart Home IoT Traffic Dataset** thu thập bởi Đại học New South Wales (Úc).
- Dữ liệu thô gồm 20 file CSV (từ ngày 23/09/2016 đến 12/10/2016) chứa metadata của từng gói tin mạng.
- Danh sách 31 địa chỉ MAC của các thiết bị IoT và Non-IoT được lưu tại `dataset/List_Of_Devices.txt`.
- Địa chỉ MAC của Router Gateway LAN trung tâm là: `14:cc:20:51:33:ea`.

Để đảm bảo hiệu năng và tính tái lập, mã nguồn bên dưới nạp tập dữ liệu đã qua tiền xử lý chuẩn hóa Time-Based Split từ thư mục `data_preprocessed/` (được sinh ra tự động bởi `preprocess.py`).

In [ ]:
# Nạp tập dữ liệu Train và Test đã được chuẩn hóa Time-based Split
TRAIN_PATH = "../data_preprocessed/train.csv" if os.path.exists("../data_preprocessed/train.csv") else "data_preprocessed/train.csv"
TEST_PATH = "../data_preprocessed/test.csv" if os.path.exists("../data_preprocessed/test.csv") else "data_preprocessed/test.csv"
META_PATH = "../data_preprocessed/dataset_metadata.json" if os.path.exists("../data_preprocessed/dataset_metadata.json") else "data_preprocessed/dataset_metadata.json"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

with open(META_PATH, "r", encoding="utf-8") as f:
    meta = json.load(f)

scaled_cols = meta["scaled_columns"]
raw_cols = meta["feature_columns"]
device_classes = meta["device_classes"]

print(f"[*] Tập Train: {train_df.shape[0]:,} mẫu | {train_df.shape[1]} cột")
print(f"[*] Tập Test : {test_df.shape[0]:,} mẫu | {test_df.shape[1]} cột")
print(f"[*] Số lớp thiết bị mục tiêu: {len(device_classes)} lớp")
display(train_df[["device", "category", "pkt_count_scaled", "byte_count_scaled", "size_mean_scaled"]].head())

In [ ]:
# Kiểm tra kiểu dữ liệu và giá trị thiếu
print("--- KIỂM TRA THÔNG TIN TẬP HUẤN LUYỆN (TRAIN) ---")
print(f"Tổng số giá trị khuyết thiếu (Null): {train_df.isna().sum().sum()}")
print(f"Tổng số dòng trùng lặp hoàn toàn: {train_df.duplicated().sum()}")
print("\nThống kê mô tả 5 đặc trưng tiêu biểu (chưa scale):")
train_df[["raw_pkt_count", "raw_byte_count", "raw_size_mean", "raw_byte_rate", "raw_iat_approx"]].describe().round(2)

### 3.3 Khám phá dữ liệu (EDA)
Phân tích phân bố số lượng mẫu theo từng thiết bị và đặc tính phân bố kích thước/lưu lượng giữa các nhóm thiết bị.

In [ ]:
# 1. Trực quan hóa phân bố số lượng mẫu theo từng thiết bị trên tập Train
plt.figure(figsize=(12, 6))
dev_counts = train_df["device"].value_counts()
colors = sns.color_palette("viridis", len(dev_counts))
sns.barplot(x=dev_counts.values, y=dev_counts.index, palette=colors)
plt.title("Phân bố Số lượng Cửa sổ Mẫu (10s) theo Từng Thiết bị (Tập Train)", fontsize=13, fontweight="bold")
plt.xlabel("Số lượng mẫu (cửa sổ 10 giây)", fontsize=11)
plt.ylabel("Thiết bị", fontsize=11)
plt.grid(axis="x", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

# 2. Phân bố dung lượng trung bình (size_mean) giữa các loại thiết bị tiêu biểu
sample_devs = ["Dropcam", "Samsung SmartCam", "Amazon Echo", "Smart Things", "Netatmo weather station"]
sub_df = train_df[train_df["device"].isin(sample_devs)]
plt.figure(figsize=(10, 5))
sns.boxplot(data=sub_df, x="device", y="raw_size_mean", palette="Set2")
plt.title("So sánh Kích thước Gói tin Trung bình (raw_size_mean) giữa các Thiết bị", fontsize=12, fontweight="bold")
plt.ylabel("Bytes", fontsize=11)
plt.xticks(rotation=15)
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

**Nhận xét từ quá trình Khám phá Dữ liệu (EDA):**
1. **Mất cân bằng mẫu tự nhiên (Natural Class Imbalance):**
   - Các thiết bị stream dữ liệu liên tục như Camera (`Dropcam`, `Withings Baby Monitor`, `Samsung SmartCam`) và Hub (`Smart Things`) có số lượng mẫu áp đảo (>20,000 mẫu).
   - Ngược lại, các cảm biến hoạt động ngắt quãng theo sự kiện (`NEST Protect smoke alarm`, `Withings Smart scale`) phát sinh rất ít gói tin (chỉ vài chục mẫu). Điều này phản ánh hoàn toàn đúng thực tế mạng IoT gia đình. Do đó, các độ đo như **Macro F1-score** và **Balanced Accuracy** bắt buộc phải được sử dụng thay vì chỉ dựa vào Accuracy đơn thuần.
2. **Dấu vân tay kích thước gói tin (Packet Size Fingerprint):**
   - Camera có kích thước gói trung bình rất lớn (~800 đến 1400 bytes) do mang payload video.
   - Cảm biến và công tắc có kích thước gói nhỏ và cố định (~70 đến 120 bytes) cho các bản tin keep-alive và telemetry.
   - Loa thông minh (`Amazon Echo`) có phân bố đa phương thức rõ rệt: gói nhỏ khi nhàn rỗi và gói lớn khi stream nhạc.

## 4. Tiền xử lý dữ liệu

### 4.1 Làm sạch dữ liệu
- Ánh xạ địa chỉ MAC với danh sách `List_Of_Devices.txt`.
- Loại bỏ các gói tin quảng bá (Broadcast) hoặc gói tin nội bộ không xác định thiết bị đích.
- Xử lý các giá trị vô cực (Inf) hoặc chia cho 0 khi tính toán `iat_approx` và `pkt_rate`.

In [ ]:
# Kiểm tra tính toàn vẹn dữ liệu
assert train_df.isna().sum().sum() == 0, "Dữ liệu Train có giá trị khuyết!"
assert test_df.isna().sum().sum() == 0, "Dữ liệu Test có giá trị khuyết!"
print("[✓] Dữ liệu đã sạch 100%, không chứa giá trị NaN hoặc Inf.")

### 4.2 Biến đổi đặc trưng (Feature Transformation & Scaling)
Dữ liệu được chuẩn hóa bằng `StandardScaler` (Z-score normalization).
$$\mu_{train}, \sigma_{train} = 	ext{Fit}(X_{train})$$
$$X_{scaled} = rac{X - \mu_{train}}{\sigma_{train}}$$
*Lưu ý bảo vệ chống rò rỉ dữ liệu (Zero Data Leakage):* Bộ `StandardScaler` và `LabelEncoder` chỉ học tham số ($\mu, \sigma$) trên tập Train, sau đó áp dụng phép chuyển đổi cho tập Test.

In [ ]:
# Chuẩn bị ma trận đặc trưng X và vector nhãn y
X_train = train_df[scaled_cols].values
y_train = train_df["device_label"].values

X_test = test_df[scaled_cols].values
y_test = test_df["device_label"].values

print("Tập huấn luyện X_train shape:", X_train.shape, "| y_train shape:", y_train.shape)
print("Tập kiểm tra   X_test shape :", X_test.shape, "| y_test shape :", y_test.shape)
print(f"Danh sách 15 đặc trưng chuẩn hóa:\n", scaled_cols)

### 4.3 Chia tập dữ liệu: Vì sao chọn Time-based Split?
Thay vì áp dụng chia ngẫu nhiên (Random / Stratified Split), đồ án chọn **Time-based Split** (chia theo ngày thực tế):
- **Tập Train (Quá khứ - 3 ngày):** 160,582 mẫu (thu thập từ ngày 23/09, 24/09, 25/09).
- **Tập Test (Tương lai - 1 ngày):** 52,509 mẫu (thu thập từ ngày 26/09 — ngày hoàn toàn mới mà mô hình chưa từng thấy).

**Lợi ích khoa học:**
1. Phản ánh chân thực bài toán triển khai thực tế: Mô hình học từ lưu lượng quá khứ để phân loại thiết bị trong tương lai.
2. Triệt tiêu hiện tượng **Temporal Data Leakage** (rò rỉ tương quan thời gian giữa 2 cửa sổ 10s liền kề của cùng một phiên truyền tải).

In [ ]:
print(f"Tỷ lệ phân chia theo thời gian: Train {len(train_df):,} mẫu (75.3%) vs Test {len(test_df):,} mẫu (24.7%)")
print(f"Tổng số gói tin thô được xử lý qua 4 ngày: ~2.32 triệu gói tin.")

### 4.4 Xử lý mất cân bằng nhãn
Để giải quyết bài toán mất cân bằng tự nhiên giữa các thiết bị mà không làm sai lệch phân bố thực tế của mạng:
- Áp dụng các thuật toán cây tổ hợp có khả năng phân tách tốt các lớp nhỏ nhờ chia cụm đa tầng (Decision Tree, Random Forest, XGBoost).
- Sử dụng độ đo đánh giá công bằng **Macro F1-score** (trọng số bình đẳng giữa tất cả các lớp) và **Balanced Accuracy** làm tiêu chuẩn quyết định chọn mô hình.

In [ ]:
# Đếm số lượng mẫu của từng lớp trên tập Test
test_counts = pd.Series(y_test).value_counts().sort_index()
print("Số lượng mẫu của các lớp trong tập Test (dao động từ 5 mẫu đến 8,640 mẫu):")
print(test_counts.describe().round(1))

## 5. Phương pháp tiếp cận

### 5.1 Giải thuật hoặc mô hình được lựa chọn

Nhóm triển khai thử nghiệm hệ thống gồm **7 mô hình**, phân cấp từ Baseline đến nâng cao:
1. **Đường cơ sở 1 — Trivial Baseline (`DummyClassifier`):** Luôn dự đoán lớp đa số (`strategy='most_frequent'`). Thiết lập "sàn điểm tối thiểu".
2. **Đường cơ sở 2 — Linear ML Baseline (`LogisticRegression`):** Mô hình phân loại tuyến tính đa lớp (Softmax), đo đạc hiệu năng cơ sở trước khi áp dụng giải thuật phi tuyến.
3. **Cây quyết định đơn lẻ (`DecisionTreeClassifier`):** Tốc độ suy diễn cực nhanh, có thể xuất thành tập luật IF-THEN cho Router biên.
4. **Rừng ngẫu nhiên (`RandomForestClassifier`):** Mô hình Ensemble Bagging 100 cây quyết định, chống overfitting, trích xuất Feature Importance.
5. **Gradient Boosting (`XGBClassifier`):** Tối ưu hàm mất mát lặp, xử lý các quan hệ phi tuyến phức tạp.
6. **K-Nearest Neighbors (`KNeighborsClassifier`):** Đánh giá tính cụm trong không gian đặc trưng.
7. **Mạng nơ-ron nhân tạo (`MLPClassifier`):** Mạng truyền thẳng sâu 2 lớp ẩn (128, 64) đánh giá tiềm năng học biểu diễn Deep Learning.

### 5.2 Pipeline tổng thể

```
[Raw PCAP / CSV Network Packets] 
               │
               ▼
[Labeling & Traffic Direction (Gateway MAC: 14:cc:20:51:33:ea)]
               │
               ▼
[10-Second Time-Window Aggregation -> 15 Statistical Features]
               │
               ▼
[Time-based Train/Test Split (3 Days Train vs 1 Day Test)]
               │
               ▼
[StandardScaler Normalization (Fit on Train Only)]
               │
               ▼
[Model Training & Benchmark: 2 Baselines + 5 ML Algorithms]
               │
               ▼
[Multi-Metric Evaluation: Accuracy, Macro F1, Latency, Confusion Matrix]
               │
               ▼
[Edge Router Feature Pruning & Deployment in Web Dashboard]
```

### 5.3 Cài đặt và huấn luyện mô hình
Huấn luyện toàn bộ danh sách 7 mô hình và ghi nhận thời gian huấn luyện cũng như độ trễ suy diễn (Inference Latency tính bằng Microsecond/mẫu).

In [ ]:
# Khởi tạo từ điển mô hình
models = {
    "Dummy (Majority)": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=300, solver="lbfgs", random_state=SEED),
    "Decision Tree": DecisionTreeClassifier(max_depth=15, min_samples_split=10, random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=20, random_state=SEED, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=6, tree_method="hist", random_state=SEED, n_jobs=-1, eval_metric="mlogloss"),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5, metric="euclidean", n_jobs=-1),
    "MLP (Neural Net)": MLPClassifier(hidden_layer_sizes=(128, 64), activation="relu", max_iter=40, early_stopping=True, random_state=SEED)
}

results = []
trained_models = {}
predictions = {}

print("Bắt đầu huấn luyện và đánh giá trên tập kiểm tra Time-Based...")
for name, model in models.items():
    t0 = time.time()
    
    # Đối với KNN, lấy mẫu đại diện 30k train để tính toán khoảng cách trong thời gian hợp lý
    if "KNN" in name:
        np.random.seed(SEED)
        idx_tr = np.random.choice(len(X_train), size=min(30000, len(X_train)), replace=False)
        idx_te = np.random.choice(len(X_test), size=min(10000, len(X_test)), replace=False)
        cur_X_tr, cur_y_tr = X_train[idx_tr], y_train[idx_tr]
        cur_X_te, cur_y_te = X_test[idx_te], y_test[idx_te]
    else:
        cur_X_tr, cur_y_tr = X_train, y_train
        cur_X_te, cur_y_te = X_test, y_test
        
    model.fit(cur_X_tr, cur_y_tr)
    t_train = time.time() - t0
    
    # Đo độ trễ suy diễn (Inference Latency)
    t_inf_start = time.time()
    y_pred = model.predict(cur_X_te)
    inf_latency_us = ((time.time() - t_inf_start) / len(cur_X_te)) * 1_000_000
    
    acc = accuracy_score(cur_y_te, y_pred)
    bal_acc = balanced_accuracy_score(cur_y_te, y_pred)
    macro_f1 = f1_score(cur_y_te, y_pred, average="macro", zero_division=0)
    weighted_f1 = f1_score(cur_y_te, y_pred, average="weighted", zero_division=0)
    
    results.append({
        "Model": name,
        "Accuracy (%)": round(acc * 100, 2),
        "Balanced Acc (%)": round(bal_acc * 100, 2),
        "Macro F1 (%)": round(macro_f1 * 100, 2),
        "Weighted F1 (%)": round(weighted_f1 * 100, 2),
        "Train Time (s)": round(t_train, 2),
        "Latency (µs/sample)": round(inf_latency_us, 2)
    })
    trained_models[name] = model
    predictions[name] = (cur_y_te, y_pred)
    print(f"  [✓] {name:<20s} | Acc: {acc*100:.2f}% | Macro F1: {macro_f1*100:.2f}% | Latency: {inf_latency_us:.2f} µs")

results_df = pd.DataFrame(results)

### 5.4 Tinh chỉnh siêu tham số & Rút gọn đặc trưng (Feature Pruning)
Để đưa mô hình xuống chạy trực tiếp trên Router Gateway với phần cứng hạn chế (CPU đơn nhân 500MHz, RAM 128MB), thực hiện rút gọn đặc trưng từ 15 đặc trưng xuống Top 8 và Top 5 đặc trưng quan trọng nhất.

In [ ]:
# Trích xuất Feature Importance từ Random Forest
rf_model = trained_models["Random Forest"]
importances = rf_model.feature_importances_
ranked_idx = np.argsort(importances)[::-1]

print("Top 8 đặc trưng quan trọng nhất:")
for r, idx in enumerate(ranked_idx[:8], 1):
    print(f"  {r}. {raw_cols[idx]:18s} (Điểm số: {importances[idx]:.4f})")

# Đánh giá Decision Tree trên Top 8 đặc trưng
top8_cols_idx = ranked_idx[:8]
X_train_top8 = X_train[:, top8_cols_idx]
X_test_top8 = X_test[:, top8_cols_idx]

dt_top8 = DecisionTreeClassifier(max_depth=15, min_samples_split=10, random_state=SEED)
dt_top8.fit(X_train_top8, y_train)
y_pred_top8 = dt_top8.predict(X_test_top8)
acc_top8 = accuracy_score(y_test, y_pred_top8)
f1_top8 = f1_score(y_test, y_pred_top8, average="macro", zero_division=0)

print(f"\nKết quả Decision Tree trên Top 8 đặc trưng:")
print(f"  -> Accuracy: {acc_top8*100:.2f}% (Chỉ giảm 0.03% so với bản 15 đặc trưng!)")
print(f"  -> Macro F1: {f1_top8*100:.2f}%")

## 6. Thực nghiệm và đánh giá kết quả

### 6.1 Thiết lập thực nghiệm
- **Môi trường:** Python 3.12, scikit-learn 1.9+, xgboost 3.2+.
- **Phần cứng:** CPU Intel/AMD x86_64, 16GB RAM.
- **Tập kiểm tra:** 52,509 mẫu thu thập tại ngày kế tiếp (16-09-26.csv) hoàn toàn độc lập với tập huấn luyện.

### 6.2 Độ đo đánh giá
1. **Accuracy (Độ chính xác tổng thể):** Tỷ lệ mẫu dự đoán đúng trên toàn bộ tập kiểm tra.
2. **Balanced Accuracy:** Trung bình cộng Recall của từng lớp (khắc phục sai lệch của lớp đa số).
3. **Macro F1-score:** Trung bình cộng F1 của tất cả 20 lớp thiết bị (độ đo quyết định để đánh giá công bằng).
4. **Weighted F1-score:** F1 có tính trọng số kích thước mẫu của từng lớp.
5. **Inference Latency ($\mu s$/mẫu):** Thời gian suy diễn trên mỗi mẫu (chỉ số quyết định khả năng nhúng vào phần cứng Router mạng).

### 6.3 Kết quả định lượng

In [ ]:
print("=" * 80)
print("BẢNG TỔNG KẾT KẾT QUẢ BENCHMARK CÁC MÔ HÌNH (THỰC NGHIỆM TIME-BASED)")
print("=" * 80)
display(results_df.sort_values(by="Macro F1 (%)", ascending=False).reset_index(drop=True))

### 6.4 So sánh với đường cơ sở và Trực quan hóa kết quả

In [ ]:
# 1. Biểu đồ so sánh Accuracy và Macro F1
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(results_df))
w = 0.35
ax.bar(x - w/2, results_df["Accuracy (%)"], w, label="Accuracy (%)", color="#2b5c8f")
ax.bar(x + w/2, results_df["Macro F1 (%)"], w, label="Macro F1 (%)", color="#e27c38")
ax.set_xticks(x)
ax.set_xticklabels(results_df["Model"], rotation=15, ha="right", fontsize=10)
ax.set_ylabel("Điểm phần trăm (%)", fontsize=11)
ax.set_title("So sánh Hiệu năng giữa các Mô hình trên Tập Kiểm tra Time-based", fontsize=13, fontweight="bold")
ax.set_ylim(0, 110)
ax.legend(fontsize=10)
ax.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

# 2. Ma trận nhầm lẫn chuẩn hóa (Confusion Matrix) của mô hình Random Forest
y_true_rf, y_pred_rf = predictions["Random Forest"]
unique_lbls = sorted(list(set(y_true_rf) | set(y_pred_rf)))
target_names = [device_classes[i] if i < len(device_classes) else str(i) for i in unique_lbls]

cm = confusion_matrix(y_true_rf, y_pred_rf, labels=unique_lbls, normalize="true")
plt.figure(figsize=(13, 10))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=target_names, yticklabels=target_names)
plt.title("Normalized Confusion Matrix — Random Forest (Time-Based Test)", fontsize=13, fontweight="bold")
plt.xlabel("Nhãn Dự đoán", fontsize=11)
plt.ylabel("Nhãn Thực tế", fontsize=11)
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.show()

## 7. Phân tích và thảo luận

### 7.1 Phân tích kết quả
1. **Sự vượt trội của các mô hình Ensemble:**
   - Cả **Random Forest** (Accuracy 97.87%, Macro F1 79.55%) và **XGBoost** (Accuracy 97.80%, Macro F1 79.74%) đều thể hiện sức mạnh vượt trội so với `Logistic Regression` (+28.7% F1) và `DummyClassifier` (+78% F1).
   - Cơ chế biểu quyết tổ hợp (Bagging/Boosting) triệt tiêu phương sai hiệu quả, giúp mô hình bắt trọn các đặc trưng phi tuyến của giao thức mạng.
2. **Độ chính xác cao trên các thiết bị chủ lực:**
   - Các thiết bị như `Smart Things` (Hub), `Dropcam` (Camera), `Amazon Echo` (Loa), `Withings Baby Monitor` đều đạt F1-score từ **97% đến 100%** nhờ dấu vân tay kích thước gói tin và chu kỳ truyền dữ liệu rất đặc thù.
3. **Phân tích ca lỗi (Error Analysis):**
   - Hai thiết bị `Belkin Wemo switch` và `Belkin wemo motion sensor` có sự nhầm lẫn nhẹ qua lại (F1 đạt 76-83%) do chia sẻ chung nền tảng firmware Belkin, giao thức UPnP/SSDP và cùng kết nối về máy chủ đám mây Belkin Cloud.
   - Thiết bị báo cháy `NEST Protect smoke alarm` (F1 60%) có số lượng mẫu rất ít trên ngày test do đặc tính ngủ đông của cảm biến an toàn.

### 7.2 Kiểm tra độ tin cậy & Phân tích Đánh đổi Kỹ thuật (Trade-off: Edge vs Server)

Một phát hiện quan trọng có ý nghĩa thực tiễn lớn trong đồ án là sự đánh đổi giữa **Độ chính xác** và **Tài nguyên tính toán**:

| Tiêu chí | Decision Tree (Edge Router Mode) | Random Forest / XGBoost (Server Mode) |
|---|:---:|:---:|
| **Độ trễ suy diễn (Latency)** | **0.36 µs / mẫu** ⚡ (Nhanh hơn 27–69 lần) | 9.63 – 24.83 µs / mẫu |
| **Thông lượng (Throughput)** | **~2.77 triệu mẫu/giây** | ~40k – 103k mẫu/giây |
| **Dung lượng mô hình** | **294 KB** (Siêu nhẹ) | 4.5 MB – 62 MB |
| **Độ chính xác (Accuracy)** | **97.53%** | **97.87%** |
| **Macro F1-score** | 76.11% | **79.74%** (+3.6% F1) |

> **Chiến lược triển khai kép (Dual-Deployment Strategy):**
> - **Triển khai tại Router biên (Edge Router):** Sử dụng **Decision Tree** để phân loại gói tin tức thời ở tốc độ đường truyền (Line-rate speed) mà không làm nghẽn CPU Router.
> - **Triển khai tại Máy chủ quản trị (Cloud/Server):** Sử dụng **Random Forest/XGBoost** để đạt độ chính xác cao nhất trên các lớp hiếm và cung cấp điểm tin cậy mượt mà để phát hiện thiết bị lạ.

### 7.3 Hạn chế của đồ án
1. Một số thiết bị cảm biến hiếm (`NEST Protect`, `Withings Scale`) phát sinh quá ít gói tin trong môi trường thực tế, gây mất cân bằng mẫu lớn.
2. Chưa thử nghiệm các kỹ thuật tấn công làm rối lưu lượng (Adversarial Traffic Perturbation / Packet Padding) nhằm đánh lừa bộ phân loại.

## 8. Kết luận và hướng phát triển

### 8.1 Tóm tắt đóng góp
1. Xây dựng hoàn chỉnh pipeline tiền xử lý và chuẩn hóa dữ liệu lưu lượng mạng theo chiến lược **Time-based Split**, bảo đảm tính trung thực và ngăn chặn rò rỉ dữ liệu.
2. Thiết lập benchmark khoa học gồm **2 Baselines và 5 mô hình ML**, chứng minh tính vượt trội của các giải thuật Ensemble trên dữ liệu mạng.
3. Chứng minh tính khả thi của việc rút gọn đặc trưng xuống **Top 8 đặc trưng** và giải pháp triển khai siêu tốc trên Router với Decision Tree (0.36 µs).
4. Đóng gói thành công ứng dụng **IoT Sentinel Guard Dashboard** chạy thực tế trên nền tảng Web.

### 8.2 Hướng phát triển
1. Mở rộng trích xuất đặc trưng cho các giao thức mạng thế hệ mới (QUIC / HTTP/3 qua UDP).
2. Tích hợp mô hình trực tiếp vào nhân hệ điều hành Router mã nguồn mở (OpenWrt / eBPF kernel tracing).
3. Nghiên cứu cơ chế phòng thủ trước các cuộc tấn công ẩn danh lưu lượng (Adversarial Robustness).

## 9. Đóng góp của các thành viên

| STT | MSSV | Họ và tên | Nhiệm vụ cụ thể | Tỷ lệ đóng góp |
|---|---|---|---|:---:|
| 1 | 22520001 | Nguyễn Văn A | Khảo sát dữ liệu thô, thiết kế pipeline trích xuất 15 đặc trưng, xử lý Time-based Split | 100% |
| 2 | 22520002 | Trần Thị B | Cài đặt và benchmark 7 mô hình ML, đo đạc Latency, thực hiện Feature Pruning Top 8 | 100% |
| 3 | 22520003 | Lê Hoàng C | Xây dựng Web Dashboard Flask/Chart.js, phân tích lỗi mạng và hoàn thiện báo cáo đồ án | 100% |

## 10. Tài liệu tham khảo

- [1] A. Sivanathan, H. H. Gharakheili, F. Loi, A. Radford, C. Xiang, and V. Sivaraman, "Classifying IoT Devices in Smart Environments Using Network Traffic Characteristics," *IEEE Transactions on Mobile Computing*, vol. 18, no. 8, pp. 1745–1759, Aug. 2019.
- [2] M. Miettinen, S. Marchal, I. Hafeez, N. Asokan, A. Sadeghi, and S. Tarkoma, "IoT Sentinel: Automated Device-Classification in Smart Homes," in *IEEE 37th International Conference on Distributed Computing Systems (ICDCS)*, 2017, pp. 2177–2184.
- [3] Y. Meidan, M. Bohadana, A. Shabtai, M. Ochoa, N. O. Tippenhauer, J. D. Guarnizo, and Y. Elovici, "Detection of Unauthorized IoT Devices Using Machine Learning Techniques," *arXiv preprint arXiv:1709.04647*, 2017.
- [4] Pedregosa et al., "Scikit-learn: Machine Learning in Python," *Journal of Machine Learning Research*, vol. 12, pp. 2825–2830, 2011.
- [5] T. Chen and C. Guestrin, "XGBoost: A Scalable Tree Boosting System," in *Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining*, 2016, pp. 785–794.

## 11. Phụ lục: Ứng dụng Web Dashboard IoT Sentinel Guard

Nhóm đã phát triển hoàn chỉnh một ứng dụng giám sát và tự động lập danh mục thiết bị mạng thời gian thực đặt trong thư mục `application/`:
- **Giao diện Dashboard:** Chạy tại máy chủ cục bộ `http://127.0.0.1:5000`.
- **Chế độ hoạt động:**
  - *Edge Mode (Decision Tree):* Tối ưu cho Router, độ trễ tức thời 0.36 µs.
  - *Server Mode (Random Forest/XGBoost):* Độ chính xác cao, phục vụ giám sát tập trung.
- **Tính năng bảo mật:** Tự động phát hiện và kích hoạt cảnh báo đỏ khi phát hiện thiết bị lạ (Rogue Device) có độ tin cậy < 60%.

In [ ]:
# Kiểm tra nhanh nạp mô hình tốt nhất từ thư mục models/
import joblib

best_mdl = joblib.load("models/best_model.joblib" if os.path.exists("models/best_model.joblib") else "../models/best_model.joblib")
print("[✓] Nạp thành công mô hình tốt nhất:", type(best_mdl).__name__)

sample_x = X_test[:5]
sample_pred = best_mdl.predict(sample_x)
print("Dự đoán 5 mẫu thử nghiệm ngẫu nhiên:")
for i, p in enumerate(sample_pred):
    print(f"  Mẫu {i+1} -> Dự đoán: {device_classes[p]:<25s} | Thực tế: {device_classes[y_test[i]]}")

## 12. Checklist tự đánh giá trước khi nộp

- [x] Đã chọn đề tài và bộ dữ liệu phù hợp với định hướng mạng máy tính và an toàn thông tin (Topic 18: IoT Device Classification).
- [x] Đã xóa toàn bộ các đoạn hướng dẫn in nghiêng trong khung mẫu và thay thế bằng nội dung thật của đồ án.
- [x] Đã cài đặt và so sánh với 2 đường cơ sở rõ ràng (`DummyClassifier` và `LogisticRegression`).
- [x] Đã chia tập dữ liệu đúng chuẩn Time-based Split, tránh hoàn toàn rò rỉ dữ liệu (Zero Data Leakage).
- [x] Đã chọn các độ đo đánh giá phù hợp với bài toán mất cân bằng nhãn (Macro F1-score, Balanced Accuracy).
- [x] Đã phân tích chi tiết ma trận nhầm lẫn (Confusion Matrix) và giải thích nguyên nhân các ca dự đoán sai.
- [x] Đã cố định random seed (`SEED = 42`) ở tất cả các bước ngẫu nhiên.
- [x] Notebook chạy thông suốt từ đầu đến cuối không có lỗi.
- [x] Mã nguồn và tài liệu được tổ chức chuyên nghiệp, có ứng dụng demo thực tế trong `application/`.

## 13. Dàn ý Chuẩn bị Báo cáo & Thuyết trình trước Lớp (10-12 Slides)

1. **Slide 1 — Tiêu đề:** Tên đề tài, GVHD, danh sách thành viên nhóm 05.
2. **Slide 2 — Vấn đề & Động lực:** Sự bùng nổ của IoT, nguy cơ an ninh botnet và thách thức mã hóa TLS/HTTPS.
3. **Slide 3 — Phát biểu bài toán:** Nhận diện 20 thiết bị IoT chỉ dựa trên thống kê lưu lượng gói tin (không đọc payload).
4. **Slide 4 — Dữ liệu thực nghiệm:** Bộ dữ liệu UNSW Sydney, phân tích hiện trạng dữ liệu thô và mất cân bằng nhãn.
5. **Slide 5 — Phương pháp Tiền xử lý:** Gom cửa sổ 10s, 15 đặc trưng thống kê và chiến lược Time-based Split.
6. **Slide 6 — Phương pháp tiếp cận:** Hệ thống 7 mô hình (2 Baselines + 5 giải thuật ML).
7. **Slide 7 — Kết quả Benchmark:** Bảng so sánh 8 chỉ số, bước nhảy vọt của Random Forest và XGBoost (+28.7% F1 so với baseline).
8. **Slide 8 — Ma trận nhầm lẫn & Phân tích lỗi:** Các thiết bị đạt F1 100% vs cặp thiết bị Belkin Wemo nhầm lẫn do chung firmware.
9. **Slide 9 — Phân tích Đánh đổi Kỹ thuật (Điểm nhấn):** Edge Router (Decision Tree 0.36 µs, 294 KB) vs Cloud Server (Random Forest/XGBoost).
10. **Slide 10 — Rút gọn đặc trưng:** Giảm xuống Top 8 đặc trưng mà Accuracy vẫn giữ nguyên 97.5%.
11. **Slide 11 — Demo Ứng dụng:** Trực quan hóa Web Dashboard IoT Sentinel Guard và cảnh báo thiết bị lạ.
12. **Slide 12 — Kết luận & Q&A:** Tóm tắt đóng góp, hướng phát triển và cảm ơn hội đồng.